In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [14]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("gpt-5.5")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [12]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

I didn’t water it today—I'm not there in person to do it. I can help you stay on top of it, though.

Would you like me to set up a daily watering reminder and a quick log? For example:
- Reminder time: 10:30 am daily
- Log: date watered, soil moisture note
- Quick care tip: check if the top inch of soil is dry before watering, and water until it drains.

If you tell me the plant type and your preferred times, I can tailor the reminder.


In [5]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


In [15]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="Once every two weeks with a diluted fertilizer should be sufficient."),
        HumanMessage(content="Do you think it needs to be repotted soon?"),
     ]}
)

print(response["messages"][-1].content)

Probably not immediately, but we should check the drainage holes—if roots are poking out or the soil is drying out unusually fast, it may be time to repot. Otherwise, we can wait until it’s actively growing more, likely in spring or early summer.


In [16]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5.5-2026-04-23
